# MetaJudge: Metacognitive Calibration Benchmark

**Competition:** Kaggle — Measuring Progress Toward AGI (Metacognition Track)  
**Version:** v0.5.5.1 (clean subset: 105 items)  
**Metric:** Brier-derived calibration score (1 − Brier, strictly proper)  

This notebook defines the official MetaJudge calibration benchmark task.
It uses the Kaggle Benchmarks SDK to evaluate a single model against
the clean 105-item calibration set. Model selection is handled via the
Kaggle Task Detail UI / Add Models mechanism.

For the full 5-model comparative analysis, see the companion
narrative notebook: `metajudge_narrative.ipynb`.

In [ ]:
# Cell 1 — Imports & Setup
import sys, os
from dataclasses import dataclass

# Make metajudge package importable from Kaggle Benchmarks dataset
_pkg_paths = [
    "/kaggle/input/datasets/seanmcgee2025/metajudge-benchmark",
    "/kaggle/input/metajudge-benchmark",
]
for _p in _pkg_paths:
    if os.path.exists(_p):
        sys.path.insert(0, _p)
        break

import kaggle_benchmarks as kbench
from kaggle_benchmarks import chats, assertions

from metajudge.notebook_helpers import (
    resolve_data_root, load_benchmark_dataset, load_clean_manifest,
    get_excluded_item_ids, filter_clean_subset, build_answer_key,
)
from metajudge.scoring.adjudication import brier_item_score
from metajudge.scoring.grading_v2 import grade_item, load_registry

print("MetaJudge benchmark notebook loaded.")

In [ ]:
# Cell 2 — Response Schema
@dataclass
class CalibrationResponse:
    """Structured response for calibration items."""
    answer: str = ""
    confidence: float = 0.5
    reason_for_uncertainty: str = ""
    would_verify_if_possible: bool = False

    def __init__(self, **kwargs):
        fields = {f.name for f in self.__dataclass_fields__.values()}
        for k, v in kwargs.items():
            if k in fields:
                setattr(self, k, v)
            # Silently drop unexpected fields (models sometimes misspell keys)
        for f in self.__dataclass_fields__.values():
            if not hasattr(self, f.name):
                setattr(self, f.name, f.default)

In [ ]:
# Cell 3 — Load Clean Dataset & Registry
import pandas as pd

data_root = resolve_data_root()
print(f"Data root: {data_root}")

# Load full dataset, then filter to clean subset
all_items = load_benchmark_dataset(data_root)
manifest = load_clean_manifest(data_root)
cal_excluded, _ = get_excluded_item_ids(manifest)
clean_items = filter_clean_subset(all_items, cal_excluded)

dataset = pd.DataFrame(clean_items)
print(f"Clean calibration set: {len(dataset)} items "
      f"(excluded {len(cal_excluded)} suspect items from {len(all_items)})")

# Load grading registry
REGISTRY = load_registry(os.path.join(data_root, "adjudication_registry.json"))
print(f"Registry: {len(REGISTRY)} items loaded")

In [ ]:
# Cell 4 — Single-Item Task Definition

@kbench.task(name="metacog_calibration_v1")
def metacog_calibration(llm, item_id: str, question: str,
                        gold_answer: str, mechanism_primary: str) -> float:
    """MetaJudge Confidence Calibration — single item evaluation.
    
    Prompts the model for an answer + confidence score, then computes
    the Brier-derived calibration score: 1 - (confidence - outcome)^2.
    
    This is a strictly proper scoring rule (Brier 1950).
    """
    with chats.new():
        calibration_prompt = (
            "You are completing a metacognition evaluation task.\n\n"
            "Task: Confidence Calibration\n"
            f"Question:\n{question}\n\n"
            "Instructions:\n"
            "1. Put only your final answer in the `answer` field.\n"
            "2. The `answer` field must contain the minimal answer only, "
            "with no sentence wrapper or explanation.\n"
            "3. If the question requests a number only, return only the number.\n"
            "4. If the question requests one word only, return only that word.\n"
            "5. Provide a confidence score from 0.0 to 1.0.\n"
            "6. Explain why you are or are not certain in `reason_for_uncertainty`.\n"
            "7. Say whether you would verify this if possible.\n\n"
            "Return valid structured output with keys: "
            "answer, confidence, reason_for_uncertainty, would_verify_if_possible"
        )
        response = llm.prompt(calibration_prompt, schema=CalibrationResponse)

    conf = max(0.0, min(1.0, response.confidence))
    assertions.assert_true(
        0.0 <= response.confidence <= 1.0,
        expectation=f"Confidence {response.confidence} must be in [0, 1]"
    )

    is_correct = grade_item(item_id, response.answer, REGISTRY)["correct"]
    score = brier_item_score(is_correct, conf)

    print(f"  [{item_id}] answer={response.answer!r}, "
          f"conf={conf:.2f}, correct={is_correct}, score={score:.4f}")

    return score

In [ ]:
# Cell 5 — Batch Task (official benchmark entry point)

@kbench.task(name="metacog_calibration_clean_v1")
def metacog_calibration_batch(llm, df) -> float:
    """MetaJudge Calibration Benchmark — clean 105-item evaluation.
    
    Runs the full clean calibration set and returns the headline score:
    mean of per-item Brier-derived scores (1 - Brier). Higher is better.
    
    This is the official MetaJudge calibration metric.
    """
    eval_cols = ["item_id", "question", "gold_answer", "mechanism_primary"]
    eval_df = df[eval_cols].copy()

    with kbench.client.enable_cache():
        runs = metacog_calibration.evaluate(
            stop_condition=lambda runs: len(runs) == eval_df.shape[0],
            max_attempts=1,
            llm=[llm],
            evaluation_data=eval_df,
            n_jobs=3,
        )

    eval_results = runs.as_dataframe()
    scores = eval_results["result"].astype(float)
    headline = float(scores.mean())

    print(f"\n{'='*50}")
    print(f"MetaJudge Calibration Results (clean set)")
    print(f"{'='*50}")
    print(f"Items evaluated: {len(scores)}")
    print(f"Headline score (mean 1-Brier): {headline:.4f}")
    print(f"Score range: [{float(scores.min()):.4f}, {float(scores.max()):.4f}]")
    print(f"{'='*50}")

    return headline

# Run the benchmark
headline_score = metacog_calibration_batch.run(kbench.llm, dataset)
print(f"\nFinal headline score: {headline_score.result}")

In [ ]:
# Cell 6 — Minimal Audit Output
import json as _json

output_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "outputs"
os.makedirs(output_dir, exist_ok=True)

audit_summary = {
    "benchmark_version": "v0.5.5.1",
    "dataset": "clean_subset",
    "n_items": len(dataset),
    "n_excluded": len(cal_excluded),
    "headline_score": headline_score.result if hasattr(headline_score, 'result') else headline_score,
}

with open(os.path.join(output_dir, "benchmark_run_summary.json"), "w") as f:
    _json.dump(audit_summary, f, indent=2)

print(f"Audit summary saved to {output_dir}/benchmark_run_summary.json")

In [ ]:
%choose metacog_calibration_clean_v1